# Colab Setup

**For Colab users:** Uncomment and run the cell below to mount Drive.  
**For local users:** Skip this cell.

In [ ]:
# Uncomment for Colab:
"""
from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path

# Set project root for Colab
PROJECT_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/Final_Project_Deep_Learning')

# Add to Python path so imports work
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Change working directory
import os
os.chdir(PROJECT_ROOT)

print(f"Working directory: {os.getcwd()}")
print(f"Python path includes: {PROJECT_ROOT}")
"""

## Setup: Dependency Verification and Installation

This cell checks for the presence of essential Python libraries (like `numpy`, `torch`, `musdb`, `nbformat`, etc.).
If any required library is not found, it attempts to install it automatically using `pip`.
It also verifies the availability of PyTorch with CUDA, which is crucial for GPU-accelerated training.

In [ ]:
from models.utils import verify_and_install_packages, check_pytorch_cuda_status

packages = ['numpy', 'matplotlib', 'librosa', 'tqdm', 'sklearn', 'stempeg', 
            'torch', 'torchvision', 'torchaudio', 'musdb', 'transformers']

verify_and_install_packages(packages)
check_pytorch_cuda_status()

## Imports and Environment Setup

- Import required libraries (torch, numpy, matplotlib, etc.)

- Set device (CPU/GPU)

In [ ]:
import sys
from pathlib import Path
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from IPython.display import Audio, display
import gc

import models.utils as utils
from models import models as ma

PROJECT_ROOT, DATA_DIR, CHECKPOINT_DIR, device, IN_COLAB = utils.setup_project_environment()
utils.set_seed(42)

## MUSDB18 Setup

**Quick Start:**
1. Download MUSDB18
2. Extract to project folder as `musdb18/`
3. Run preprocessing below

**Expected structure:**
```
musdb18/
  train/     (~100 songs)
  valid/     (~16 songs)
  test/      (~50 songs)
```

In [ ]:
SUBMISSION_MODE = False
if not SUBMISSION_MODE:
    MUSDB18_PATH = utils.verify_musdb18_dataset(PROJECT_ROOT)
    mix_files_stage1 = mix_files_stage2 = tgt_files_stage1 = tgt_files_stage2 = []

## Data Preprocessing

Preprocess MUSDB18 into 8-second waveform chunks for training, validation, and testing.

In [ ]:
skip_data_processing = True
FOLDERS_TO_EXTRACT = ['vocals'] # choose which folders to extract: ['stage1', 'stage2', 'vocals']
EXTRACT_DATA_SUB_ON_COLAB = False  # Set to True to extract data_sub.zip into local data directory when running on colab
SAMPLE_RATE = 22050
CHUNK_DURATION = 8.0
CHUNK_OVERLAP = 4.0

if skip_data_processing:
    print("Data processing skipped (skip_data_processing = True)")
    print("Using pre-existing data directories...")
else:
    DATA_DIR = utils.extract_and_process_data(
        PROJECT_ROOT, 
        DATA_DIR, 
        MUSDB18_PATH, 
        IN_COLAB, 
        folders_to_extract=FOLDERS_TO_EXTRACT,
        sample_rate=SAMPLE_RATE,
        chunk_duration=CHUNK_DURATION,
        chunk_overlap=CHUNK_OVERLAP
    )

if EXTRACT_DATA_SUB_ON_COLAB and IN_COLAB:
    utils.extract_data_sub_to_local(PROJECT_ROOT, DATA_DIR)

## Two Architectures for Comparison

**Model1 (LSTM)** Sequential bidirectional LSTM with masking output.

**Model2 (U-Net)** 2D CNN encoder-decoder with skip connections.

In [ ]:
lstm_preview, _, _, _ = utils.initialize_model_a_lstm(device)
unet_preview, _, _, _ = utils.initialize_model_a_unet(device)

lstm_params = sum(p.numel() for p in lstm_preview.parameters())
unet_params = sum(p.numel() for p in unet_preview.parameters())

print("\nModel 1 (LSTM):")
print(f"   Parameters: {lstm_params:,}")
print(f"   Type: Bidirectional LSTM with masking")

print("\nModel 2 (U-Net):")
print(f"   Parameters: {unet_params:,}")
print(f"   Type: 2D CNN encoder-decoder")

del lstm_preview, unet_preview
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Train Both Models - Stage 1

Sequential training: LSTM first, then U-Net

In [ ]:
SKIP_TRAINING_STAGE1 = False
CHUNK_DURATION = 8.0
LSTM_BATCH_SIZE = 128
UNET_BATCH_SIZE = 32

results = utils.train_stage1_models(
    data_dir=DATA_DIR,
    checkpoint_dir=CHECKPOINT_DIR,
    device=device,
    chunk_duration=CHUNK_DURATION,
    skip_training=SKIP_TRAINING_STAGE1,
    lstm_batch_size=LSTM_BATCH_SIZE,
    unet_batch_size=UNET_BATCH_SIZE
)

model_lstm = results['model_lstm']
processor_lstm = results['processor_lstm']
loss_fn_lstm = results['loss_fn_lstm']
model_unet = results['model_unet']
processor_unet = results['processor_unet']
loss_fn_unet = results['loss_fn_unet']
hist_lstm_s1 = results['hist_lstm_s1']
hist_unet_s1 = results['hist_unet_s1']
ckpt_lstm_s1 = results['ckpt_lstm_s1']
ckpt_unet_s1 = results['ckpt_unet_s1']

## Stage 1 Test Evaluation

Compute loss on held-out test set (forward pass only) to assess generalization.


In [ ]:
test_lstm_s1, test_unet_s1 = utils.evaluate_stage1_models(
    model_lstm=model_lstm,
    processor_lstm=processor_lstm,
    loss_fn_lstm=loss_fn_lstm,
    model_unet=model_unet,
    processor_unet=processor_unet,
    loss_fn_unet=loss_fn_unet,
    ckpt_lstm_s1=ckpt_lstm_s1,
    ckpt_unet_s1=ckpt_unet_s1,
    data_dir=DATA_DIR,
    checkpoint_dir=CHECKPOINT_DIR,
    chunk_duration=CHUNK_DURATION,
    device=device
)

utils.plot_stage1_training_curves(
    hist_lstm_s1=hist_lstm_s1,
    hist_unet_s1=hist_unet_s1,
    test_lstm_s1=test_lstm_s1,
    test_unet_s1=test_unet_s1,
    checkpoint_dir=CHECKPOINT_DIR,
    chunk_duration=CHUNK_DURATION
)

## Stage 1 Evaluation (Spectrograms + Audio)

Compare LSTM vs U-Net separation quality on Stage 1 test samples.

In [ ]:
print("="*70)
print("STAGE 1 EVALUATION: Simplified Mixture (Vocals+Other) → Other")
print("="*70)

SR = 22050
DURATION = 120.0
CHUNK_LEN = 8.0
HOP_LENGTH = 4.0

mix_wav, tgt_wav, has_target, selected = utils.load_and_stitch_test_chunks(
    data_dir=DATA_DIR,
    stage='stage1',
    sr=SR,
    duration=DURATION,
    hop_length=HOP_LENGTH,
    auto_select=False
)

est_lstm_s1, est_unet_s1 = utils.evaluate_and_visualize_stage1(
    model_lstm=model_lstm,
    processor_lstm=processor_lstm,
    model_unet=model_unet,
    processor_unet=processor_unet,
    mix_wav=mix_wav,
    tgt_wav=tgt_wav,
    has_target=has_target,
    selected_song=selected,
    sr=SR,
    chunk_len=CHUNK_LEN,
    device=device
)

## Train Both Models - Stage 2

Curriculum step: Stage 2 uses 4→1 channels and continues from Stage 1 weights.

In [ ]:
STAGE2_ENABLED = True
SKIP_TRAINING_STAGE2 = False

LSTM_BATCH_SIZE_S2 = 128
UNET_BATCH_SIZE_S2 = 32

result_s2 = utils.train_stage2_models(
    data_dir=DATA_DIR,
    checkpoint_dir=CHECKPOINT_DIR,
    device=device,
    chunk_duration=CHUNK_DURATION,
    skip_training=SKIP_TRAINING_STAGE2,
    lstm_batch_size=LSTM_BATCH_SIZE_S2,
    unet_batch_size=UNET_BATCH_SIZE_S2,
    stage2_enabled=STAGE2_ENABLED
)

model_lstm = result_s2['model_lstm']
processor_lstm = result_s2['processor_lstm']
loss_fn_lstm = result_s2['loss_fn_lstm']
model_unet = result_s2['model_unet']
processor_unet = result_s2['processor_unet']
loss_fn_unet = result_s2['loss_fn_unet']
hist_lstm_s2 = result_s2['hist_lstm_s2']
hist_unet_s2 = result_s2['hist_unet_s2']
ckpt_lstm_s2 = result_s2['ckpt_lstm_s2']
ckpt_unet_s2 = result_s2['ckpt_unet_s2']

## Stage 2 Test Evaluation

Compute loss on held-out test set (forward pass only) to assess generalization.


In [ ]:
test_lstm_s2, test_unet_s2 = utils.evaluate_stage2_models(
    ckpt_lstm_s2=ckpt_lstm_s2,
    ckpt_unet_s2=ckpt_unet_s2,
    data_dir=DATA_DIR,
    checkpoint_dir=CHECKPOINT_DIR,
    chunk_duration=CHUNK_DURATION,
    device=device
)

utils.plot_stage2_training_curves(
    hist_lstm_s2=hist_lstm_s2,
    hist_unet_s2=hist_unet_s2,
    test_lstm_s2=test_lstm_s2,
    test_unet_s2=test_unet_s2,
    checkpoint_dir=CHECKPOINT_DIR,
    chunk_duration=CHUNK_DURATION,
    stage2_enabled=STAGE2_ENABLED
)

## Stage 2 Evaluation (Spectrograms + Audio)

Compare LSTM vs U-Net separation quality on Stage 2 test samples.

In [ ]:
print("="*70)
print("STAGE 2 EVALUATION: Full Band Mixture → Vocals")
print("="*70)

SR = 22050
DURATION = 120.0
CHUNK_LEN = 8.0
HOP_LENGTH = 4.0

mix_wav, tgt_wav, has_target, selected = utils.load_and_stitch_test_chunks(
    data_dir=DATA_DIR,
    stage='stage2',
    sr=SR,
    duration=DURATION,
    hop_length=HOP_LENGTH,
    auto_select=False
)

est_lstm_s2, est_unet_s2 = utils.evaluate_and_visualize_stage2(
    model_lstm=model_lstm,
    processor_lstm=processor_lstm,
    model_unet=model_unet,
    processor_unet=processor_unet,
    mix_wav=mix_wav,
    tgt_wav=tgt_wav,
    has_target=has_target,
    selected_song=selected,
    sr=SR,
    chunk_len=CHUNK_LEN,
    device=device
)

## Quantitative Evaluation

Compute BSS metrics (SDR/SIR/SAR) on test set using museval library.

In [ ]:
print("model1 = LSTM")
print("model2 = U-Net")
print("\n")

NUM_TEST_SAMPLES = 500
STAGE_FOR_EVAL = "stage2"
RANDOM_SAMPLE_TEST_CHUNKS = True
RANDOM_SEED = 42
FORCE_RECOMPUTE_METRICS = False

metrics_ckpt = CHECKPOINT_DIR / f"quant_metrics_unet_vs_lstm{STAGE_FOR_EVAL}_{NUM_TEST_SAMPLES}samples.pkl"
print(f"Metrics checkpoint: {metrics_ckpt}")

if FORCE_RECOMPUTE_METRICS and metrics_ckpt.exists():
    metrics_ckpt.unlink()
    print("Removed existing metrics checkpoint (force recompute enabled).")

metrics = utils.evaluate_separation_quality(
    model_1=model_lstm,
    model_2=model_unet,
    processor_1=processor_lstm,
    processor_2=processor_unet,
    test_data_dir=DATA_DIR,
    stage=STAGE_FOR_EVAL,
    num_samples=NUM_TEST_SAMPLES,
    sr=22050,
    device=device,
    save_path=metrics_ckpt,
    load_if_exists=True,
    random_sampling=RANDOM_SAMPLE_TEST_CHUNKS,
    random_seed=RANDOM_SEED
)

## Custom Song Inference (Upload)

Upload a song (place it in the folder "data/user_uploads") and run inference with both models. This is the final step.

In [ ]:
utils.handle_user_upload_and_inference(
    data_dir=DATA_DIR,
    model_lstm=model_lstm,
    model_unet=model_unet,
    processor_lstm=processor_lstm,
    processor_unet=processor_unet,
    device=device,
    in_colab=IN_COLAB,
    sr=SR,
    duration=None,
    unet_batch_size=16
)

# Adding Attention To our Unet
In this part we will try to improve seperation by adding attention to our Unet

**Model3 (Unettention)** Unet with attention between encoder and decoder

In [ ]:
unetattention_preview = ma.UNetAttention(
    in_channels=1, out_channels=1, base_filters=32, 
    num_layers=5, num_heads=4
).to(device)

attn_params = sum(p.numel() for p in unetattention_preview.parameters())

print("\nModel 3 (UNetAttention):")
print(f"   Parameters: {attn_params:,}")
print(f"   Type: U-Net with Multi-Head Self-Attention at bottleneck")

del unetattention_preview
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Train Unettention Model - Stage 1

In [ ]:
skip_attn_s1 = False

results_attn_s1 = utils.train_unetattention_stage1(
    data_dir=DATA_DIR,
    checkpoint_dir=CHECKPOINT_DIR,
    device=device,
    chunk_duration=CHUNK_DURATION,
    skip_training=skip_attn_s1,
    batch_size=32,
    base_filters=32,
    num_layers=4,
    num_heads=4
)

model_attn_s1 = results_attn_s1['model_attn_s1']
processor_attn = results_attn_s1['processor_attn']
loss_fn_attn = results_attn_s1['loss_fn_attn']
hist_attn_s1 = results_attn_s1['hist_attn_s1']
ckpt_attn_s1 = results_attn_s1['ckpt_attn_s1']

## Compare Training Results (Stage 1)

Side-by-side comparison of Standard U-Net vs. UNetAttention training curves. We look for faster convergence or lower final validation loss to see if the attention mechanism helps.

In [ ]:
utils.compare_unet_vs_unetattention_stage1(
    checkpoint_dir=CHECKPOINT_DIR,
    chunk_duration=CHUNK_DURATION,
    hist_unet_s1=hist_unet_s1,
    hist_attn_s1=hist_attn_s1
)

## Stage 1 Visual Evaluation (Spectrograms + Audio)

Visual inspection of the Attention model's separation quality on a test sample. We look for clearer separation in the spectrograms compared to previous models.

In [ ]:
SR = 22050
DURATION = 120.0
CHUNK_LEN = 8.0
HOP_LENGTH = 4.0

mix_wav, tgt_wav, has_target, selected = utils.load_and_stitch_test_chunks(
    data_dir=DATA_DIR,
    stage='stage1',
    sr=SR,
    duration=DURATION,
    hop_length=HOP_LENGTH,
    auto_select=False
)

est_unet, est_attn = utils.evaluate_and_visualize_unet_vs_unetattention_stage1(
    model_unet=model_unet,
    processor_unet=processor_unet,
    model_attn=model_attn_s1,
    processor_attn=processor_attn,
    mix_wav=mix_wav,
    tgt_wav=tgt_wav,
    has_target=has_target,
    selected_song=selected,
    sr=SR,
    chunk_len=CHUNK_LEN,
    device=device
)

## Train UNetAttention - Stage 2

In [ ]:
skip_attn_s2 = False

results_attn_s2 = utils.train_unetattention_stage2(
    data_dir=DATA_DIR,
    checkpoint_dir=CHECKPOINT_DIR,
    device=device,
    chunk_duration=CHUNK_DURATION,
    skip_training=skip_attn_s2,
    batch_size=32,
    base_filters=32,
    num_layers=4,
    num_heads=4,
    ckpt_attn_s1=ckpt_attn_s1
)

model_attn_s2 = results_attn_s2['model_attn_s2']
processor_attn_s2 = results_attn_s2['processor_attn_s2']
loss_fn_attn_s2 = results_attn_s2['loss_fn_attn_s2']
hist_attn_s2 = results_attn_s2['hist_attn_s2']
ckpt_attn_s2 = results_attn_s2['ckpt_attn_s2']

## Compare Training Results (Stage 2)

Side-by-side comparison of Standard U-Net vs. UNetAttention training curves. We look for faster convergence or lower final validation loss to see if the attention mechanism helps.

In [ ]:
utils.compare_unet_vs_unetattention_stage2(
    checkpoint_dir=CHECKPOINT_DIR,
    chunk_duration=CHUNK_DURATION,
    hist_unet_s2=hist_unet_s2,
    hist_attn_s2=hist_attn_s2
)

## Stage 2 Visual Evaluation (Spectrograms + Audio)

Visual inspection of the Attention model's separation quality on a test sample. We look for clearer separation in the spectrograms compared to previous models.

In [ ]:
SR = 22050
DURATION = 120.0
CHUNK_LEN = 8.0
HOP_LENGTH = 4.0

mix_wav, tgt_wav, has_target, selected = utils.load_and_stitch_test_chunks(
    data_dir=DATA_DIR,
    stage='stage2',
    sr=SR,
    duration=DURATION,
    hop_length=HOP_LENGTH,
    auto_select=False
)

est_unet_s2, est_attn_s2 = utils.evaluate_and_visualize_unet_vs_unetattention_stage2(
    model_unet=model_unet,
    processor_unet=processor_unet,
    model_attn=model_attn_s2,
    processor_attn=processor_attn_s2,
    mix_wav=mix_wav,
    tgt_wav=tgt_wav,
    has_target=has_target,
    selected_song=selected,
    sr=SR,
    chunk_len=CHUNK_LEN,
    device=device
)

## Custom Song Inference (Upload)

Upload a song (place it in the folder "data/user_uploads") and run inference with both models. This is the final step.

In [ ]:
utils.handle_user_upload_unetattention_inference(
    data_dir=DATA_DIR,
    model_unet=model_unet,
    model_attn=model_attn_s2,
    processor_unet=processor_unet,
    processor_attn=processor_attn_s2,
    device=device,
    in_colab=IN_COLAB,
    sr=22050,
    duration=None,
    chunk_len=8.0
)

## Quantitative Evaluation

Compute BSS metrics (SDR/SIR/SAR) on test set using museval library.

In [ ]:
print("model1 = U-Net")
print("model2 = UNetAttention")
print("\n")

NUM_TEST_SAMPLES = 500
STAGE_FOR_EVAL = "stage2"
RANDOM_SAMPLE_TEST_CHUNKS = True
RANDOM_SEED = 42
FORCE_RECOMPUTE_METRICS = False

metrics_ckpt = CHECKPOINT_DIR / f"quant_metrics_unet_vs_unetattention_{STAGE_FOR_EVAL}_{NUM_TEST_SAMPLES}samples.pkl"
print(f"Metrics checkpoint: {metrics_ckpt}")



if FORCE_RECOMPUTE_METRICS and metrics_ckpt.exists():
    metrics_ckpt.unlink()
    print("Removed existing metrics checkpoint (force recompute enabled).")


metrics_raw = utils.evaluate_separation_quality(
    model_1=model_unet,
    model_2=model_attn_s2,
    processor_1=processor_unet,
    processor_2=processor_attn_s2,
    test_data_dir=DATA_DIR,
    stage=STAGE_FOR_EVAL,
    num_samples=NUM_TEST_SAMPLES,
    sr=22050,
    device=device,
    save_path=metrics_ckpt,
    load_if_exists=True,
    random_sampling=RANDOM_SAMPLE_TEST_CHUNKS,
    random_seed=RANDOM_SEED
)

# Training a unettention model to extract vocals only

In [ ]:
skip_voc_training = False

results_voc = utils.train_unetattention_vocals_only(
    data_dir=DATA_DIR,
    checkpoint_dir=CHECKPOINT_DIR,
    device=device,
    skip_training=skip_voc_training,
    batch_size=4,
    base_filters=32,
    num_layers=5,
    num_heads=4,
    learning_rate=1e-4,
    num_epochs=20
)

model_voc_attn = results_voc['model_voc_attn']
processor_voc = results_voc['processor_voc']
loss_fn_voc = results_voc['loss_fn_voc']
voc_history = results_voc['voc_history']
ckpt_voc_attn = results_voc['ckpt_voc_attn']

## Verify Training Success - UNetAttention

In [ ]:
SR = 22050
DURATION = 120.0
CHUNK_LEN = 8.0
HOP_LENGTH = 4.0

result = utils.evaluate_vocals_extraction_unetattention(
    data_dir=DATA_DIR,
    checkpoint_dir=CHECKPOINT_DIR,
    device=device,
    model_voc_attn=model_voc_attn if 'model_voc_attn' in locals() else None,
    processor_voc=processor_voc if 'processor_voc' in locals() else None,
    sr=SR,
    duration=DURATION,
    chunk_len=CHUNK_LEN,
    hop_length=HOP_LENGTH,
    base_filters=32,
    num_layers=5,
    num_heads=4,
    auto_select=False
)

extracted_vocals = result['extracted_vocals']
mix_wav = result['mix_wav']
vocal_wav_gt = result['vocal_wav_gt']